
# 09 — Influence de la profondeur historique sur l'entraînement

## A — Présentation et exploration des données

### A.1 — Contexte et objectifs

Les expérimentations précédentes ont permis de valider la faisabilité d'une prévision de consommation énergétique à plusieurs horizons, de H+1 à H+24, à partir de modèles de Machine Learning.

Les résultats obtenus ont notamment montré l'intérêt d'une Random Forest globale, exploitant les caractéristiques temporelles, les consommations historiques et les informations propres aux sites.

Toutefois, une question reste ouverte : **quelle profondeur historique est nécessaire pour entraîner efficacement notre modèle ?**

Cette expérimentation vise à mesurer l'influence de la quantité de données d'entraînement sur la qualité des prévisions.

Quatre fenêtres historiques seront comparées :

- 30 jours ;
- 90 jours ;
- 180 jours ;
- 365 jours.

Afin de rendre cette comparaison pertinente, nous conserverons une architecture de modèle, des caractéristiques et des hyperparamètres identiques. Les fenêtres d'entraînement partageront une même date de fin et seront évaluées sur une période de validation commune.

### A.2 — Source des données

Les données utilisées proviennent de l'API EnerVision, via l'endpoint de récupération des readings.

Une collecte couvrant la période du 1er janvier 2025 au 30 juin 2026 inclus a été réalisée pour les sept sites disponibles.

Cette collecte représente :

- 546 jours ;
- 7 sites ;
- 3 822 requêtes API ;
- 1 100 736 mesures ;
- une granularité théorique de cinq minutes.

Les réponses brutes ont été conservées sous forme de fichiers JSON individuels, organisés par site et par journée.

Contrairement à l'expérimentation sur le dataset historique CSV, cette étude repose sur les readings issus de l'API destinée à alimenter le futur pipeline de données.

### A.3 — Démarche expérimentale

L'expérimentation suivra cinq étapes :

1. Contrôler la qualité, la couverture et la continuité des readings.
2. Agréger les mesures à une granularité horaire.
3. Construire un dataset supervisé pour les horizons H+1 à H+24.
4. Comparer les performances selon la durée de l'historique d'entraînement.
5. Évaluer la configuration retenue sur une période de test indépendante.

L'objectif est d'identifier un compromis pertinent entre volume d'entraînement, performances prédictives et coût de calcul.

Les conclusions resteront limitées aux données et aux sites étudiés.



#### Procédure de collecte

Les readings ont été récupérés à l'aide du script `scripts/collect_readings_month.py`, qui interroge l'API EnerVision pour chaque site et chaque journée.

La commande suivante a été exécutée depuis le répertoire `ml/`, avec l'environnement virtuel Python activé :

```powershell
python scripts/collect_readings_month.py --start 2025-01-01 --end 2026-07-01
```

Les paramètres `--start` et `--end` définissent respectivement les bornes inclusive et exclusive de la période de collecte, en UTC.

Le script utilise une limite de 288 mesures par requête, correspondant à une journée complète avec une mesure toutes les cinq minutes.

Il effectue également plusieurs contrôles :

- Vérification du type de réponse de l'API.
- Validation de l'appartenance des timestamps à la période demandée.
- Détection des doublons de timestamps.
- Signalement des journées potentiellement incomplètes.
- Écriture atomique des fichiers JSON.
- Conservation des fichiers existants pour permettre la reprise d'une collecte interrompue.

Les données sont enregistrées dans `data/raw/readings/`, sous la forme `SITE001_20250101.json`.

**Résultat de la collecte :** 3 822 requêtes réussies, 1 100 736 mesures récupérées, aucune requête échouée et aucune journée signalée comme incomplète.

Les fichiers bruts sont conservés indépendamment des transformations effectuées dans les notebooks.


## Imports et configuration

In [31]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

# Chemins du projet
DATA_DIR = Path("../data/raw/readings")

# Configuration de l'expérimentation
EXPECTED_SITES = 7
EXPECTED_READINGS_PER_DAY = 288
EXPECTED_DAYS = 546

# Vérification de la présence des données
assert DATA_DIR.exists(), f"Répertoire introuvable : {DATA_DIR}"

json_files = sorted(DATA_DIR.glob("SITE*.json"))

print(f"Fichiers JSON détectés : {len(json_files):,}")
print(f"Fichiers attendus : {EXPECTED_SITES * EXPECTED_DAYS:,}")
print(f"Répertoire : {DATA_DIR.resolve()}")

Fichiers JSON détectés : 3,822
Fichiers attendus : 3,822
Répertoire : C:\Users\duche\OneDrive\Documents\GitHub\smart_energy_optimizer\ml\data\raw\readings



### A.4 — Chargement et consolidation des readings

La collecte des données EnerVision a produit 3 822 fichiers JSON, chacun correspondant aux mesures d'un site pour une journée donnée.

Avant toute transformation, ces fichiers doivent être consolidés dans un dataset unique afin de faciliter leur exploration.

Cette étape consiste à :

- Lire les fichiers JSON issus de la collecte.
- Extraire les mesures et leurs métadonnées.
- Associer chaque mesure à son site d'origine.
- Convertir les timestamps au format datetime UTC.
- Vérifier le volume et la structure du dataset obtenu.

Les données de consommation ne seront pas modifiées à ce stade. L'objectif est de disposer d'une représentation consolidée des readings bruts, avant les contrôles de qualité et l'agrégation horaire.


In [32]:
# Chargement et consolidation des readings.
records = []

for file_path in json_files:
    with file_path.open("r", encoding="utf-8") as file:
        payload = json.load(file)

    site_id = payload["site_id"]

    for reading in payload["readings"]:
        records.append(
            {
                **reading,
                "site_id": site_id,
            }
        )

df_readings = pd.DataFrame.from_records(records)

# Conversion des timestamps en UTC.
df_readings["timestamp"] = pd.to_datetime(
    df_readings["timestamp"],
    utc=True,
    errors="coerce",
)

# Tri chronologique par site.
df_readings = df_readings.sort_values(["site_id", "timestamp"]).reset_index(drop=True)

print(f"Nombre total de readings : {len(df_readings):,}")
print(f"Nombre de sites : {df_readings['site_id'].nunique()}")
print(f"Début : {df_readings['timestamp'].min()}")
print(f"Fin : {df_readings['timestamp'].max()}")

print("\nColonnes disponibles :")
print(df_readings.columns.tolist())

display(df_readings.head())

Nombre total de readings : 1,100,736
Nombre de sites : 7
Début : 2025-01-01 00:00:00+00:00
Fin : 2026-06-30 23:55:00+00:00

Colonnes disponibles :
['timestamp', 'site_id', 'site_type', 'consumption_kw', 'consumption_kwh', 'voltage_v', 'current_a', 'power_factor', 'temperature_celsius', 'humidity_percent', 'null_reasons', 'data_quality']


,timestamp,site_id,site_type,consumption_kw,consumption_kwh,voltage_v,current_a,power_factor,temperature_celsius,humidity_percent,null_reasons,data_quality
0,2025-01-01 00:00:00+00:00,SITE001,office,59.28,59.28,401.2,89.81,0.922,NaN,72.8,[temperature_sensor_failure],partial
1,2025-01-01 00:05:00+00:00,SITE001,office,57.27,57.27,396.3,87.81,0.911,10.8,70.6,[],good
2,2025-01-01 00:10:00+00:00,SITE001,office,57.22,57.22,404.3,86.02,0.972,12.0,66.1,[],good
3,2025-01-01 00:15:00+00:00,SITE001,office,62.81,62.81,399.8,95.47,0.966,10.3,74.1,[],good
4,2025-01-01 00:20:00+00:00,SITE001,office,63.24,63.24,387.3,99.25,0.897,9.6,64.6,[],good



### A.5 — Contrôle de qualité des readings

Avant de construire le dataset d'entraînement, nous procédons à un audit des mesures collectées.

Cet audit porte sur quatre dimensions :

1. **Complétude :** identification des valeurs manquantes, notamment pour la consommation électrique.
2. **Unicité :** recherche de doublons pour un même site et un même timestamp.
3. **Continuité temporelle :** vérification de la régularité des intervalles de cinq minutes.
4. **Cohérence :** identification des consommations nulles ou négatives et vérification de la répartition des mesures entre les sites.

Ces contrôles permettent de déterminer si les readings peuvent être exploités directement ou si des traitements de nettoyage sont nécessaires.

Les éventuelles anomalies seront quantifiées avant toute correction afin de préserver la traçabilité de la préparation des données.


In [33]:
# 1. Valeurs manquantes
missing_values = df_readings.isna().sum().to_frame("missing_count")

missing_values["missing_pct"] = missing_values["missing_count"] / len(df_readings) * 100

print("=== Valeurs manquantes ===")
display(missing_values.sort_values("missing_count", ascending=False))

# 2. Doublons par site et timestamp
duplicate_count = df_readings.duplicated(subset=["site_id", "timestamp"]).sum()

print(f"\nDoublons (site_id, timestamp) : {duplicate_count:,}")

# 3. Continuité temporelle
df_readings["time_diff"] = df_readings.groupby("site_id")["timestamp"].diff()

expected_interval = pd.Timedelta(minutes=5)

irregular_intervals = df_readings["time_diff"].notna() & df_readings["time_diff"].ne(
    expected_interval
)

print(f"Intervalles irréguliers : {irregular_intervals.sum():,}")

# 4. Consommations invalides
invalid_consumption = df_readings["consumption_kw"].isna() | (df_readings["consumption_kw"] <= 0)

print(f"Consommations nulles, négatives ou absentes : {invalid_consumption.sum():,}")

# 5. Répartition par site
site_summary = (
    df_readings.groupby(["site_id", "site_type"])
    .agg(
        readings=("timestamp", "count"),
        start=("timestamp", "min"),
        end=("timestamp", "max"),
        mean_consumption_kw=("consumption_kw", "mean"),
    )
    .reset_index()
)

site_summary["expected_readings"] = EXPECTED_DAYS * EXPECTED_READINGS_PER_DAY
site_summary["coverage_pct"] = site_summary["readings"] / site_summary["expected_readings"] * 100

print("\n=== Répartition par site ===")
display(site_summary)

=== Valeurs manquantes ===


,missing_count,missing_pct
temperature_celsius,98523,8.950648
humidity_percent,87007,7.904438
consumption_kw,65275,5.930123
consumption_kwh,65275,5.930123
power_factor,64966,5.902051
current_a,64966,5.902051
voltage_v,64966,5.902051
timestamp,0,0.000000
site_type,0,0.000000
site_id,0,0.000000



Doublons (site_id, timestamp) : 0
Intervalles irréguliers : 0
Consommations nulles, négatives ou absentes : 65,275

=== Répartition par site ===


,site_id,site_type,readings,start,end,mean_consumption_kw,expected_readings,coverage_pct
0,SITE001,office,157248,2025-01-01 00:00:00+00:00,2026-06-30 23:55:00+00:00,92.979398,157248,100.0
1,SITE002,factory,157248,2025-01-01 00:00:00+00:00,2026-06-30 23:55:00+00:00,547.708504,157248,100.0
2,SITE003,datacenter,157248,2025-01-01 00:00:00+00:00,2026-06-30 23:55:00+00:00,724.941536,157248,100.0
3,SITE004,retail,157248,2025-01-01 00:00:00+00:00,2026-06-30 23:55:00+00:00,230.393808,157248,100.0
4,SITE005,hospital,157248,2025-01-01 00:00:00+00:00,2026-06-30 23:55:00+00:00,468.643803,157248,100.0
5,SITE006,office,157248,2025-01-01 00:00:00+00:00,2026-06-30 23:55:00+00:00,81.422360,157248,100.0
6,SITE007,factory,157248,2025-01-01 00:00:00+00:00,2026-06-30 23:55:00+00:00,510.231965,157248,100.0




### A.6 — Analyse des indicateurs de qualité et des causes d'indisponibilité

Les readings EnerVision contiennent deux champs complémentaires : `data_quality` et `null_reasons`.

Ces informations permettent de distinguer les valeurs manquantes des éventuelles anomalies de mesure et d'identifier les causes déclarées d'indisponibilité.

Avant de procéder au nettoyage, nous analyserons :

- La répartition des différents états de qualité.
- Les causes d'indisponibilité renseignées par l'API.
- La relation entre ces indicateurs et les consommations manquantes.
- L'existence éventuelle de consommations numériques associées à un état de qualité dégradé.

Cette analyse doit permettre de définir des règles de préparation adaptées à la sémantique des données EnerVision, plutôt que d'appliquer systématiquement un traitement générique des valeurs manquantes.



In [34]:
print("=== Répartition des états de qualité ===")
display(df_readings["data_quality"].value_counts(dropna=False))

print("\n=== Répartition des causes d'indisponibilité ===")
display(df_readings["null_reasons"].astype(str).value_counts(dropna=False).head(30))

print("\n=== Qualité et disponibilité des consommations ===")

quality_summary = (
    df_readings.assign(consumption_missing=df_readings["consumption_kw"].isna())
    .groupby("data_quality", dropna=False)
    .agg(
        total_readings=("timestamp", "count"),
        missing_consumption=("consumption_missing", "sum"),
    )
)

quality_summary["missing_pct"] = (
    quality_summary["missing_consumption"] / quality_summary["total_readings"] * 100
)

display(quality_summary)

print("\n=== Exemples de mesures avec consommation absente ===")

display(
    df_readings.loc[
        df_readings["consumption_kw"].isna(),
        [
            "site_id",
            "timestamp",
            "consumption_kw",
            "data_quality",
            "null_reasons",
        ],
    ].head(15)
)

=== Répartition des états de qualité ===


data_quality
good        868398
partial     193862
critical     22205
degraded     16271
Name: count, dtype: int64


=== Répartition des causes d'indisponibilité ===


null_reasons
[]                                                                                                                      868398
['temperature_sensor_failure']                                                                                           66241
['humidity_sensor_failure']                                                                                              55533
['consumption_sensor_failure']                                                                                           36146
['electrical_sensor_failure']                                                                                            35942
['network_loss']                                                                                                         22205
['temperature_sensor_failure', 'humidity_sensor_failure']                                                                 4257
['consumption_sensor_failure', 'temperature_sensor_failure']                                      


=== Qualité et disponibilité des consommations ===


,total_readings,missing_consumption,missing_pct
data_quality,,,
critical,22205,22205,100.000000
degraded,16271,6924,42.554238
good,868398,0,0.000000
partial,193862,36146,18.645222



=== Exemples de mesures avec consommation absente ===


,site_id,timestamp,consumption_kw,data_quality,null_reasons
31,SITE001,2025-01-01 02:35:00+00:00,NaN,critical,[network_loss]
47,SITE001,2025-01-01 03:55:00+00:00,NaN,critical,[network_loss]
56,SITE001,2025-01-01 04:40:00+00:00,NaN,partial,[consumption_sensor_failure]
63,SITE001,2025-01-01 05:15:00+00:00,NaN,degraded,"[consumption_sensor_failure, temperature_senso..."
82,SITE001,2025-01-01 06:50:00+00:00,NaN,degraded,"[consumption_sensor_failure, temperature_senso..."
117,SITE001,2025-01-01 09:45:00+00:00,NaN,partial,[consumption_sensor_failure]
151,SITE001,2025-01-01 12:35:00+00:00,NaN,critical,[network_loss]
157,SITE001,2025-01-01 13:05:00+00:00,NaN,partial,[consumption_sensor_failure]
190,SITE001,2025-01-01 15:50:00+00:00,NaN,partial,[consumption_sensor_failure]
194,SITE001,2025-01-01 16:10:00+00:00,NaN,partial,[consumption_sensor_failure]



### A.7 — Interprétation des indicateurs de qualité

L'analyse des indicateurs EnerVision met en évidence quatre états de qualité : `good`, `partial`, `degraded` et `critical`.

Sur les 1 100 736 readings collectés :

- 868 398 sont classés `good` ;
- 193 862 sont classés `partial` ;
- 16 271 sont classés `degraded` ;
- 22 205 sont classés `critical`.

Les 65 275 consommations manquantes se répartissent entre les états `partial`, `degraded` et `critical`.

L'analyse des causes d'indisponibilité montre notamment des défaillances des capteurs de consommation, de température, d'humidité, des capteurs électriques ainsi que des pertes de communication réseau.

Une mesure classée `partial` ou `degraded` peut néanmoins contenir une consommation exploitable, lorsque les anomalies concernent d'autres capteurs.

**Stratégie retenue pour le POC :**

- Conserver les consommations numériques disponibles, indépendamment de l'état global de qualité.
- Exclure des calculs les consommations absentes ou non numériques.
- Conserver les timestamps des observations indisponibles.
- Préserver les indicateurs de qualité et les causes d'indisponibilité dans les données brutes.
- Contrôler la couverture des mesures valides lors de l'agrégation horaire.

Cette stratégie repose sur l'hypothèse qu'une consommation numérique reste exploitable lorsque l'anomalie concerne un autre capteur. Elle devra être confirmée par les spécifications de l'API avant une utilisation en production.


In [35]:
# Causes d'indisponibilité affectant directement la consommation.
blocking_reasons = {
    "consumption_sensor_failure",
    "network_loss",
}

has_blocking_reason = df_readings["null_reasons"].apply(
    lambda reasons: isinstance(reasons, list) and bool(blocking_reasons.intersection(reasons))
)

has_consumption = df_readings["consumption_kw"].notna()

inconsistent_readings = df_readings[has_blocking_reason & has_consumption]

print("=== Cohérence des indicateurs de panne ===")
print(f"Readings avec cause bloquante : {has_blocking_reason.sum():,}")
print(f"Readings avec cause bloquante et consommation présente : {len(inconsistent_readings):,}")

display(
    inconsistent_readings[
        [
            "site_id",
            "timestamp",
            "consumption_kw",
            "data_quality",
            "null_reasons",
        ]
    ].head(10)
)

=== Cohérence des indicateurs de panne ===
Readings avec cause bloquante : 65,275
Readings avec cause bloquante et consommation présente : 0


,site_id,timestamp,consumption_kw,data_quality,null_reasons



## B — Préparation du dataset horaire

### B.1 — Agrégation des readings

Les données EnerVision sont collectées à une fréquence de cinq minutes, soit douze mesures théoriques par heure et par site.

Pour notre modèle de prévision énergétique, nous souhaitons travailler à une granularité horaire, conformément aux expérimentations précédentes.

L'agrégation consiste à calculer la moyenne des puissances électriques `consumption_kw` disponibles pour chaque heure et chaque site.

Les contrôles précédents ont confirmé que les 65 275 consommations manquantes correspondent à des causes d'indisponibilité identifiées, notamment des pannes de capteurs et des pertes de communication.

Afin de ne pas introduire artificiellement des consommations nulles, ces valeurs manquantes sont exclues du calcul de la moyenne, tout en conservant leur présence dans le décompte des readings.

**Règle de qualité retenue :**

Une observation horaire est considérée comme exploitable lorsqu'au moins 9 des 12 readings attendus possèdent une consommation valide, soit une couverture minimale de 75 %.

Ce seuil constitue une hypothèse de préparation pour notre POC. Il permet de limiter l'utilisation de moyennes horaires reposant sur trop peu de mesures, sans exiger une disponibilité parfaite des capteurs.

Les heures ne respectant pas ce seuil conservent leur timestamp, mais leur consommation est remplacée par `NaN`.

Les indicateurs de qualité et les causes d'indisponibilité restent disponibles dans les données brutes, sans être intégrés directement aux variables prédictives.


In [36]:
# Sélection des colonnes nécessaires.
df_hourly_source = df_readings[["site_id", "site_type", "timestamp", "consumption_kw"]].copy()

# Conversion explicite en numérique.
df_hourly_source["consumption_kw"] = pd.to_numeric(
    df_hourly_source["consumption_kw"],
    errors="coerce",
)

# Pour ce POC, les consommations négatives sont invalides.
# Une consommation nulle est conservée si elle est réellement mesurée.
df_hourly_source.loc[
    df_hourly_source["consumption_kw"] < 0,
    "consumption_kw",
] = np.nan

# Agrégation par site et par heure.
df_hourly = (
    df_hourly_source.groupby(
        [
            "site_id",
            "site_type",
            pd.Grouper(key="timestamp", freq="h"),
        ]
    )
    .agg(
        consumption_kw=("consumption_kw", "mean"),
        valid_readings=("consumption_kw", "count"),
        total_readings=("timestamp", "count"),
    )
    .reset_index()
)

# Application du seuil minimal de qualité.
MIN_VALID_READINGS = 9

df_hourly.loc[
    df_hourly["valid_readings"] < MIN_VALID_READINGS,
    "consumption_kw",
] = np.nan

# Indicateur de couverture horaire.
df_hourly["coverage_pct"] = df_hourly["valid_readings"] / df_hourly["total_readings"] * 100

# Bilan.
expected_hours = EXPECTED_DAYS * 24 * EXPECTED_SITES
missing_hours = df_hourly["consumption_kw"].isna().sum()

print("=== Bilan de l'agrégation horaire ===")
print(f"Observations horaires : {len(df_hourly):,}")
print(f"Observations attendues : {expected_hours:,}")
print(f"Heures exploitables : {len(df_hourly) - missing_hours:,}")
print(f"Heures non exploitables : {missing_hours:,}")
print(f"Proportion non exploitable : {missing_hours / len(df_hourly) * 100:.2f} %")

print("\n=== Nombre de readings valides par heure ===")
display(
    df_hourly["valid_readings"]
    .value_counts()
    .sort_index()
    .rename_axis("valid_readings")
    .to_frame("nombre_heures")
)

print("\n=== Aperçu du dataset horaire ===")
display(df_hourly.head(10))

=== Bilan de l'agrégation horaire ===
Observations horaires : 91,728
Observations attendues : 91,728
Heures exploitables : 91,331
Heures non exploitables : 397
Proportion non exploitable : 0.43 %

=== Nombre de readings valides par heure ===


,nombre_heures
valid_readings,
5,1
6,1
7,35
8,360
9,2351
10,11583
11,33428
12,43969



=== Aperçu du dataset horaire ===


,site_id,site_type,timestamp,consumption_kw,valid_readings,total_readings,coverage_pct
0,SITE001,office,2025-01-01 00:00:00+00:00,59.199167,12,12,100.000000
1,SITE001,office,2025-01-01 01:00:00+00:00,58.697500,12,12,100.000000
2,SITE001,office,2025-01-01 02:00:00+00:00,60.101818,11,12,91.666667
3,SITE001,office,2025-01-01 03:00:00+00:00,60.121818,11,12,91.666667
4,SITE001,office,2025-01-01 04:00:00+00:00,60.836364,11,12,91.666667
5,SITE001,office,2025-01-01 05:00:00+00:00,59.913636,11,12,91.666667
6,SITE001,office,2025-01-01 06:00:00+00:00,60.705455,11,12,91.666667
7,SITE001,office,2025-01-01 07:00:00+00:00,58.680833,12,12,100.000000
8,SITE001,office,2025-01-01 08:00:00+00:00,79.719167,12,12,100.000000
9,SITE001,office,2025-01-01 09:00:00+00:00,112.106364,11,12,91.666667



### B.2 — Bilan de l'agrégation horaire

L'agrégation des readings EnerVision produit 91 728 observations horaires, correspondant aux 546 jours étudiés pour les sept sites.

Sur cet ensemble :

- 91 331 observations horaires sont exploitables, soit 99,57 %.
- 397 observations ne respectent pas le seuil minimal de neuf mesures valides, soit 0,43 %.
- 43 969 heures disposent de leurs douze mesures de consommation.

L'agrégation permet ainsi de réduire fortement la proportion d'observations inutilisables, initialement de 5,93 % à la granularité de cinq minutes.

Cette amélioration ne correspond pas à une reconstitution des valeurs manquantes : les moyennes horaires sont calculées uniquement à partir des mesures réellement disponibles.

Les 397 heures insuffisamment renseignées conservent leur position dans la série temporelle, avec une consommation définie à `NaN`.

Cette stratégie préserve la cohérence temporelle du dataset et permet de construire les variables historiques sans décaler artificiellement les observations.


In [37]:
# Enrichissement avec les caractéristiques calendaires.
df_hourly["hour"] = df_hourly["timestamp"].dt.hour
df_hourly["day_of_week"] = df_hourly["timestamp"].dt.dayofweek
df_hourly["month"] = df_hourly["timestamp"].dt.month
df_hourly["is_weekend"] = (df_hourly["day_of_week"] >= 5).astype(int)

# Vérification de l'unicité des observations horaires.
assert not df_hourly.duplicated(subset=["site_id", "timestamp"]).any(), (
    "Doublons détectés dans le dataset horaire."
)

# Sauvegarde intermédiaire.
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

hourly_path = PROCESSED_DIR / "readings_hourly_2025_2026.parquet"

df_hourly.to_parquet(hourly_path, index=False)

print(f"Dataset horaire sauvegardé : {hourly_path}")
print(f"Dimensions : {df_hourly.shape}")
print(f"Taille du fichier : {hourly_path.stat().st_size / 1024**2:.2f} Mo")

display(df_hourly.head())

Dataset horaire sauvegardé : ..\data\processed\readings_hourly_2025_2026.parquet
Dimensions : (91728, 11)
Taille du fichier : 0.95 Mo


,site_id,site_type,timestamp,consumption_kw,valid_readings,total_readings,coverage_pct,hour,day_of_week,month,is_weekend
0,SITE001,office,2025-01-01 00:00:00+00:00,59.199167,12,12,100.000000,0,2,1,0
1,SITE001,office,2025-01-01 01:00:00+00:00,58.697500,12,12,100.000000,1,2,1,0
2,SITE001,office,2025-01-01 02:00:00+00:00,60.101818,11,12,91.666667,2,2,1,0
3,SITE001,office,2025-01-01 03:00:00+00:00,60.121818,11,12,91.666667,3,2,1,0
4,SITE001,office,2025-01-01 04:00:00+00:00,60.836364,11,12,91.666667,4,2,1,0



### B.3 — Construction des caractéristiques historiques

Le modèle de prévision repose sur des caractéristiques décrivant l'état de la consommation au moment où une prédiction est effectuée.

Pour chaque site, nous construisons les variables suivantes :

- `lag_1h` : consommation de la dernière heure connue.
- `lag_24h` : consommation observée 24 heures avant l'origine de prévision.
- `lag_168h` : consommation observée une semaine avant l'origine de prévision.
- `rolling_mean_24h` : consommation moyenne des 24 dernières heures disponibles.
- `target_lag_168h` : consommation observée une semaine avant l'heure cible.

Les quatre premières variables décrivent l'historique disponible au moment de la prévision.

La variable `target_lag_168h` dépend de l'horizon prédit. Elle constitue une référence hebdomadaire particulièrement pertinente pour les consommations énergétiques.

Pour chaque origine de prévision, le modèle produira des exemples supervisés correspondant aux horizons H+1 à H+24.

Une attention particulière sera portée à l'alignement temporel afin qu'aucune variable prédictive ne contienne une information postérieure à l'origine de prévision.


In [38]:
# Tri préalable indispensable aux décalages temporels.
df_features = df_hourly.sort_values(["site_id", "timestamp"]).reset_index(drop=True).copy()

grouped = df_features.groupby("site_id")["consumption_kw"]

# Informations disponibles à l'origine de prévision.
# Le timestamp représente la dernière heure observée.
df_features["lag_1h"] = grouped.shift(0)
df_features["lag_24h"] = grouped.shift(24)
df_features["lag_168h"] = grouped.shift(168)

# Moyenne des 24 dernières heures, incluant l'heure d'origine.
# Une fenêtre incomplète reste manquante.
df_features["rolling_mean_24h"] = grouped.transform(
    lambda series: series.rolling(
        window=24,
        min_periods=24,
    ).mean()
)

print("=== Dataset avec caractéristiques historiques ===")
print(f"Dimensions : {df_features.shape}")

print("\n=== Valeurs manquantes ===")
display(
    df_features[
        [
            "lag_1h",
            "lag_24h",
            "lag_168h",
            "rolling_mean_24h",
        ]
    ]
    .isna()
    .sum()
    .to_frame("missing_count")
)

display(
    df_features[
        [
            "site_id",
            "timestamp",
            "consumption_kw",
            "lag_1h",
            "lag_24h",
            "lag_168h",
            "rolling_mean_24h",
        ]
    ].tail(10)
)

=== Dataset avec caractéristiques historiques ===
Dimensions : (91728, 15)

=== Valeurs manquantes ===


,missing_count
lag_1h,397
lag_24h,565
lag_168h,1570
rolling_mean_24h,9207


,site_id,timestamp,consumption_kw,lag_1h,lag_24h,lag_168h,rolling_mean_24h
91718,SITE007,2026-06-30 14:00:00+00:00,796.200833,796.200833,802.813636,797.307273,585.121006
91719,SITE007,2026-06-30 15:00:00+00:00,804.520000,804.520000,838.656667,817.568889,583.698645
91720,SITE007,2026-06-30 16:00:00+00:00,815.357273,815.357273,814.079167,817.114545,583.751899
91721,SITE007,2026-06-30 17:00:00+00:00,791.910833,791.910833,819.499167,820.245000,582.602386
91722,SITE007,2026-06-30 18:00:00+00:00,809.299000,809.299000,807.959091,809.475000,582.658215
91723,SITE007,2026-06-30 19:00:00+00:00,793.815455,793.815455,806.598000,819.909167,582.125609
91724,SITE007,2026-06-30 20:00:00+00:00,795.872727,795.872727,802.112500,793.685833,581.865618
91725,SITE007,2026-06-30 21:00:00+00:00,613.931818,613.931818,602.530000,579.870000,582.340694
91726,SITE007,2026-06-30 22:00:00+00:00,240.247500,240.247500,249.193333,244.413333,581.967951
91727,SITE007,2026-06-30 23:00:00+00:00,245.463636,245.463636,237.056667,243.591818,582.318242



### B.4 — Construction du dataset supervisé multi-horizon

À partir des observations horaires enrichies, nous construisons un dataset supervisé destiné à prédire la consommation énergétique pour les horizons H+1 à H+24.

Chaque exemple associe :

- Une origine de prévision, correspondant à la dernière heure connue.
- Un horizon de prévision compris entre 1 et 24 heures.
- Les caractéristiques historiques disponibles à cette origine.
- Les caractéristiques calendaires de l'heure cible.
- La consommation réellement observée à l'heure cible.

La variable `target_lag_168h` représente la consommation observée exactement une semaine avant l'heure cible. Pour les horizons étudiés, cette référence est toujours antérieure à l'origine de prévision.

Les exemples dont la consommation cible est indisponible seront exclus de l'entraînement et de l'évaluation, sans supprimer les heures correspondantes de la série temporelle initiale.

La construction du dataset respectera ainsi la chronologie des informations disponibles au moment de chaque prévision.


In [39]:
HORIZONS = range(1, 25)

supervised_frames = []

for horizon in HORIZONS:
    frame = df_features[
        [
            "site_id",
            "site_type",
            "timestamp",
            "lag_1h",
            "lag_24h",
            "lag_168h",
            "rolling_mean_24h",
        ]
    ].copy()

    frame = frame.rename(columns={"timestamp": "forecast_origin"})

    frame["horizon"] = horizon

    frame["target_timestamp"] = frame["forecast_origin"] + pd.Timedelta(hours=horizon)

    # Consommation cible à H+h.
    frame["target_consumption_kw"] = df_features.groupby("site_id")["consumption_kw"].shift(
        -horizon
    )

    # Référence hebdomadaire alignée sur l'heure cible.
    # Pour H+1, il s'agit de l'heure origine -167h.
    # Pour H+24, il s'agit de l'heure origine -144h.
    frame["target_lag_168h"] = df_features.groupby("site_id")["consumption_kw"].shift(168 - horizon)

    # Caractéristiques calendaires de l'heure cible.
    frame["target_hour"] = frame["target_timestamp"].dt.hour
    frame["target_day_of_week"] = frame["target_timestamp"].dt.dayofweek
    frame["target_month"] = frame["target_timestamp"].dt.month
    frame["target_is_weekend"] = (frame["target_day_of_week"] >= 5).astype(int)

    supervised_frames.append(frame)

df_supervised = pd.concat(
    supervised_frames,
    ignore_index=True,
)

# Suppression des exemples sans cible connue ou hors période.
df_supervised = df_supervised[
    df_supervised["target_consumption_kw"].notna()
    & (df_supervised["target_timestamp"] <= df_hourly["timestamp"].max())
].reset_index(drop=True)

print("=== Dataset supervisé ===")
print(f"Nombre d'exemples : {len(df_supervised):,}")
print(f"Nombre de caractéristiques : {df_supervised.shape[1]}")
print(f"Horizons : {df_supervised['horizon'].min()} à {df_supervised['horizon'].max()}")

print("\n=== Exemples par horizon ===")
display(df_supervised.groupby("horizon").size().to_frame("nombre_exemples"))

print("\n=== Valeurs manquantes des caractéristiques ===")
display(
    df_supervised[
        [
            "lag_1h",
            "lag_24h",
            "lag_168h",
            "rolling_mean_24h",
            "target_lag_168h",
        ]
    ]
    .isna()
    .sum()
    .to_frame("missing_count")
)

display(df_supervised.head())

=== Dataset supervisé ===
Nombre d'exemples : 2,189,893
Nombre de caractéristiques : 15
Horizons : 1 à 24

=== Exemples par horizon ===


,nombre_exemples
horizon,
1,91324
2,91317
3,91310
4,91304
5,91297
6,91290
7,91283
8,91277
9,91270



=== Valeurs manquantes des caractéristiques ===


,missing_count
lag_1h,9480
lag_24h,13471
lag_168h,37538
rolling_mean_24h,219888
target_lag_168h,35461


,site_id,site_type,forecast_origin,lag_1h,lag_24h,lag_168h,rolling_mean_24h,horizon,target_timestamp,target_consumption_kw,target_lag_168h,target_hour,target_day_of_week,target_month,target_is_weekend
0,SITE001,office,2025-01-01 00:00:00+00:00,59.199167,NaN,NaN,NaN,1,2025-01-01 01:00:00+00:00,58.697500,NaN,1,2,1,0
1,SITE001,office,2025-01-01 01:00:00+00:00,58.697500,NaN,NaN,NaN,1,2025-01-01 02:00:00+00:00,60.101818,NaN,2,2,1,0
2,SITE001,office,2025-01-01 02:00:00+00:00,60.101818,NaN,NaN,NaN,1,2025-01-01 03:00:00+00:00,60.121818,NaN,3,2,1,0
3,SITE001,office,2025-01-01 03:00:00+00:00,60.121818,NaN,NaN,NaN,1,2025-01-01 04:00:00+00:00,60.836364,NaN,4,2,1,0
4,SITE001,office,2025-01-01 04:00:00+00:00,60.836364,NaN,NaN,NaN,1,2025-01-01 05:00:00+00:00,59.913636,NaN,5,2,1,0



## C — Influence de la profondeur historique

### C.1 — Protocole expérimental

Cette expérimentation vise à mesurer l'effet de la durée de l'historique d'entraînement sur les performances d'une Random Forest globale.

Quatre profondeurs historiques sont étudiées : 30, 90, 180 et 365 jours.

Pour isoler l'effet de la quantité de données, les quatre modèles utiliseront :

- Le même algorithme et les mêmes hyperparamètres.
- Les mêmes caractéristiques prédictives.
- Les mêmes règles de préparation et de traitement des valeurs manquantes.
- La même période de validation.

Les fenêtres d'entraînement sont emboîtées et se terminent toutes au 31 décembre 2025.

Le découpage chronologique est le suivant :

- **Entraînement :** fenêtres historiques se terminant le 31 décembre 2025.
- **Validation :** du 1er janvier au 31 mars 2026.
- **Test indépendant :** du 1er avril au 30 juin 2026.

Les exemples supervisés sont sélectionnés en tenant compte à la fois de l'origine de prévision et de la date cible. Aucun exemple d'entraînement ne doit posséder une cible appartenant à la période de validation ou de test.

Le jeu de test restera inutilisé pendant la comparaison des fenêtres historiques. Il servira uniquement à l'évaluation finale de la configuration retenue.

Les références historiques nécessaires au calcul des variables prédictives pourront précéder le début de chaque fenêtre d'entraînement, sans être utilisées comme exemples supervisés supplémentaires.


#### Construction des fenêtres

In [40]:
# Bornes chronologiques, en UTC.
TRAIN_END = pd.Timestamp("2026-01-01", tz="UTC")
VALIDATION_END = pd.Timestamp("2026-04-01", tz="UTC")
TEST_END = pd.Timestamp("2026-07-01", tz="UTC")

TRAINING_WINDOWS = [30, 90, 180, 365]

# Stratégie B retenue dans le notebook 08 :
# conserver uniquement les exemples disposant de la référence hebdomadaire.
df_eligible = df_supervised.loc[df_supervised["target_lag_168h"].notna()].copy()

# Validation commune à toutes les fenêtres.
df_validation = df_eligible.loc[
    (df_eligible["forecast_origin"] >= TRAIN_END)
    & (df_eligible["target_timestamp"] < VALIDATION_END)
].copy()

# Test indépendant, réservé à la fin de l'expérience.
df_test = df_eligible.loc[
    (df_eligible["forecast_origin"] >= VALIDATION_END)
    & (df_eligible["target_timestamp"] < TEST_END)
].copy()

# Construction des quatre fenêtres d'entraînement.
training_sets = {}

for days in TRAINING_WINDOWS:
    window_start = TRAIN_END - pd.Timedelta(days=days)

    training_sets[days] = df_eligible.loc[
        (df_eligible["forecast_origin"] >= window_start)
        & (df_eligible["target_timestamp"] < TRAIN_END)
    ].copy()

# Bilan des volumes.
summary = pd.DataFrame(
    [
        {
            "window_days": days,
            "start": TRAIN_END - pd.Timedelta(days=days),
            "end": TRAIN_END,
            "examples": len(training_sets[days]),
            "sites": training_sets[days]["site_id"].nunique(),
        }
        for days in TRAINING_WINDOWS
    ]
)

print("=== Fenêtres d'entraînement ===")
display(summary)

print(f"\nValidation : {len(df_validation):,} exemples")
print(f"Test indépendant : {len(df_test):,} exemples")

# Vérification de l'absence de chevauchement des périodes cibles.
assert all(dataset["target_timestamp"].max() < TRAIN_END for dataset in training_sets.values())

assert df_validation["target_timestamp"].min() >= TRAIN_END
assert df_validation["target_timestamp"].max() < VALIDATION_END

assert df_test["target_timestamp"].min() >= VALIDATION_END
assert df_test["target_timestamp"].max() < TEST_END

print("\nContrôles chronologiques : OK")

=== Fenêtres d'entraînement ===


,window_days,start,end,examples,sites
0,30,2025-12-02 00:00:00+00:00,2026-01-01 00:00:00+00:00,117625,7
1,90,2025-10-03 00:00:00+00:00,2026-01-01 00:00:00+00:00,357062,7
2,180,2025-07-05 00:00:00+00:00,2026-01-01 00:00:00+00:00,717108,7
3,365,2025-01-01 00:00:00+00:00,2026-01-01 00:00:00+00:00,1430256,7



Validation : 358,260 exemples
Test indépendant : 361,778 exemples

Contrôles chronologiques : OK



### C.2 — Configuration du modèle et du prétraitement

Pour comparer l'influence de la profondeur historique, nous retenons une Random Forest globale, commune aux sept sites.

Cette approche a été sélectionnée lors des expérimentations précédentes pour son compromis entre performances, simplicité d'exploitation et capacité à représenter différents profils énergétiques.

Le modèle utilise onze caractéristiques :

- Les informations calendaires de l'heure cible.
- L'horizon de prévision.
- Les consommations historiques disponibles à l'origine de prévision.
- La consommation hebdomadaire de référence de l'heure cible.
- L'identifiant et le type de site.

Les variables numériques historiques susceptibles de contenir des valeurs manquantes sont imputées par leur médiane, calculée exclusivement sur les données d'entraînement de chaque fenêtre.

Les variables catégorielles sont encodées avec `OneHotEncoder`.

Les hyperparamètres de la Random Forest restent identiques pour les quatre fenêtres afin de limiter les facteurs de variation.

L'objectif n'est pas d'optimiser les hyperparamètres, mais de mesurer l'effet de la durée d'entraînement.


In [41]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

FEATURE_COLUMNS = [
    "horizon",
    "target_hour",
    "target_day_of_week",
    "target_month",
    "target_is_weekend",
    "lag_1h",
    "lag_24h",
    "rolling_mean_24h",
    "target_lag_168h",
    "site_id",
    "site_type",
]

NUMERIC_COLUMNS = [
    "horizon",
    "target_hour",
    "target_day_of_week",
    "target_month",
    "target_is_weekend",
    "target_lag_168h",
]

IMPUTED_COLUMNS = [
    "lag_1h",
    "lag_24h",
    "rolling_mean_24h",
]

CATEGORICAL_COLUMNS = [
    "site_id",
    "site_type",
]


def create_pipeline():
    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", "passthrough", NUMERIC_COLUMNS),
            (
                "imputed",
                SimpleImputer(strategy="median"),
                IMPUTED_COLUMNS,
            ),
            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore"),
                CATEGORICAL_COLUMNS,
            ),
        ]
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "model",
                RandomForestRegressor(
                    n_estimators=50,
                    max_depth=15,
                    min_samples_leaf=10,
                    random_state=42,
                    n_jobs=-1,
                ),
            ),
        ]
    )


print("Pipeline Random Forest configuré.")
print(f"Nombre de caractéristiques d'entrée : {len(FEATURE_COLUMNS)}")
print(f"Fenêtres à comparer : {TRAINING_WINDOWS}")

Pipeline Random Forest configuré.
Nombre de caractéristiques d'entrée : 11
Fenêtres à comparer : [30, 90, 180, 365]



### C.3 — Entraînement des modèles

Quatre Random Forest sont entraînées indépendamment sur des fenêtres historiques de 30, 90, 180 et 365 jours.

Pour chaque fenêtre :

1. Le pipeline de prétraitement est ajusté sur les exemples d'entraînement.
2. La Random Forest est entraînée avec les mêmes hyperparamètres.
3. Les prédictions sont calculées sur le jeu de validation commun.
4. Les métriques MAE, RMSE et WAPE sont calculées.
5. Le temps d'entraînement est mesuré.

Les résultats permettront d'évaluer si l'augmentation du volume historique améliore effectivement la précision des prévisions, et à quel coût de calcul.


In [42]:
from time import perf_counter

from sklearn.metrics import mean_absolute_error, mean_squared_error

X_val = df_validation[FEATURE_COLUMNS]
y_val = df_validation["target_consumption_kw"]

results = []
trained_models = {}

for days in TRAINING_WINDOWS:
    print(f"\n=== Entraînement : {days} jours ===")

    df_train = training_sets[days]

    X_train = df_train[FEATURE_COLUMNS]
    y_train = df_train["target_consumption_kw"]

    pipeline = create_pipeline()

    start_time = perf_counter()
    pipeline.fit(X_train, y_train)
    training_seconds = perf_counter() - start_time

    predictions = pipeline.predict(X_val)

    mae = mean_absolute_error(y_val, predictions)
    rmse = np.sqrt(mean_squared_error(y_val, predictions))
    wape = np.abs(y_val.to_numpy() - predictions).sum() / np.abs(y_val.to_numpy()).sum() * 100

    results.append(
        {
            "window_days": days,
            "train_examples": len(df_train),
            "validation_examples": len(df_validation),
            "mae_kw": mae,
            "rmse_kw": rmse,
            "wape_pct": wape,
            "training_seconds": training_seconds,
        }
    )

    trained_models[days] = pipeline

    print(f"Exemples d'entraînement : {len(df_train):,}")
    print(f"Durée : {training_seconds:.2f} s")
    print(f"MAE : {mae:.2f} kW")
    print(f"RMSE : {rmse:.2f} kW")
    print(f"WAPE : {wape:.2f} %")

df_results = pd.DataFrame(results).sort_values("window_days")

print("\n=== Comparaison des fenêtres ===")
display(df_results)


=== Entraînement : 30 jours ===
Exemples d'entraînement : 117,625
Durée : 1.58 s
MAE : 12.06 kW
RMSE : 19.49 kW
WAPE : 3.23 %

=== Entraînement : 90 jours ===
Exemples d'entraînement : 357,062
Durée : 5.60 s
MAE : 14.70 kW
RMSE : 23.66 kW
WAPE : 3.93 %

=== Entraînement : 180 jours ===
Exemples d'entraînement : 717,108
Durée : 13.70 s
MAE : 11.91 kW
RMSE : 20.18 kW
WAPE : 3.19 %

=== Entraînement : 365 jours ===
Exemples d'entraînement : 1,430,256
Durée : 34.08 s
MAE : 5.49 kW
RMSE : 9.20 kW
WAPE : 1.47 %

=== Comparaison des fenêtres ===


,window_days,train_examples,validation_examples,mae_kw,rmse_kw,wape_pct,training_seconds
0,30,117625,358260,12.062218,19.493917,3.226966,1.576785
1,90,357062,358260,14.702741,23.661046,3.933377,5.602162
2,180,717108,358260,11.907897,20.176174,3.185681,13.701233
3,365,1430256,358260,5.485983,9.195159,1.467647,34.075632



### C.4 — Analyse comparative des profondeurs historiques

Les quatre modèles ont été entraînés avec une architecture Random Forest identique et évalués sur une période de validation commune, de janvier à mars 2026.

Les résultats montrent une influence importante de la profondeur historique sur les performances prédictives.

Le modèle entraîné sur 365 jours obtient les meilleurs résultats :

- MAE : 5,49 kW ;
- RMSE : 9,20 kW ;
- WAPE : 1,47 % ;
- Temps d'entraînement : 32,31 secondes.

Par rapport à un entraînement sur 30 jours, la fenêtre de 365 jours réduit la MAE d'environ 54,5 %.

L'évolution des performances n'est toutefois pas monotone : la fenêtre de 90 jours produit des résultats inférieurs à celle de 30 jours.

Cette observation montre que l'ajout de données historiques ne garantit pas systématiquement une amélioration des prédictions.

L'hypothèse principale est qu'un historique annuel permet au modèle de mieux représenter la diversité des comportements énergétiques et les variations saisonnières. Cette interprétation reste toutefois à confirmer, car l'augmentation du volume d'entraînement constitue également un facteur explicatif possible.

Le coût de calcul demeure limité, avec un entraînement inférieur à une minute pour les 1,43 million d'exemples de la fenêtre annuelle.

**La profondeur de 365 jours est donc retenue pour l'évaluation finale sur le jeu de test indépendant.**

Cette sélection repose exclusivement sur les résultats de validation. Le jeu de test n'a pas été utilisé pour comparer ou sélectionner les modèles.



## D — Évaluation finale sur le jeu de test indépendant

### D.1 — Objectif et protocole

À l'issue de la comparaison des profondeurs historiques, la Random Forest entraînée sur 365 jours a été retenue en raison de ses performances sur le jeu de validation.

L'évaluation finale est réalisée sur la période du 1er avril au 30 juin 2026, qui n'a été utilisée ni pour l'entraînement ni pour la sélection du modèle.

Le modèle est comparé à deux références :

- **Baseline H-24 :** consommation observée 24 heures avant l'heure cible.
- **Baseline H-168 :** consommation observée 168 heures avant l'heure cible.

Les trois approches seront évaluées sur un ensemble commun d'observations disposant des deux références historiques.

Les métriques utilisées sont :

- MAE : erreur absolue moyenne, en kW.
- RMSE : racine de l'erreur quadratique moyenne, en kW.
- WAPE : somme des erreurs absolues rapportée à la somme des consommations réelles, en pourcentage.

Cette évaluation vise à vérifier la capacité de généralisation du modèle sur une période chronologiquement indépendante et à quantifier sa valeur ajoutée par rapport aux références historiques.


In [43]:
# Référence quotidienne alignée sur le timestamp cible.
# Elle est calculée à partir de la série horaire complète.
daily_reference = df_hourly[["site_id", "timestamp", "consumption_kw"]].rename(
    columns={
        "timestamp": "daily_reference_timestamp",
        "consumption_kw": "target_lag_24h",
    }
)

# Préparation du jeu de test avec la référence H-24.
df_test_eval = df_test.copy()

df_test_eval["daily_reference_timestamp"] = df_test_eval["target_timestamp"] - pd.Timedelta(
    hours=24
)

df_test_eval = df_test_eval.merge(
    daily_reference,
    on=["site_id", "daily_reference_timestamp"],
    how="left",
    validate="many_to_one",
)

# Comparaison sur les mêmes observations.
df_test_common = df_test_eval.dropna(subset=["target_lag_24h", "target_lag_168h"]).copy()

print("=== Couverture du test ===")
print(f"Exemples de test avec H-168 : {len(df_test):,}")
print(f"Exemples communs H-24 / H-168 : {len(df_test_common):,}")

# Prédictions du modèle retenu.
best_model = trained_models[365]

df_test_common["prediction_rf"] = best_model.predict(df_test_common[FEATURE_COLUMNS])


def calculate_metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    return {
        "mae_kw": mean_absolute_error(y_true, y_pred),
        "rmse_kw": np.sqrt(mean_squared_error(y_true, y_pred)),
        "wape_pct": (np.abs(y_true - y_pred).sum() / np.abs(y_true).sum() * 100),
    }


# Évaluation des trois approches.
y_true = df_test_common["target_consumption_kw"]

test_results = []

for name, predictions in [
    ("Baseline H-24", df_test_common["target_lag_24h"]),
    ("Baseline H-168", df_test_common["target_lag_168h"]),
    ("Random Forest 365j", df_test_common["prediction_rf"]),
]:
    test_results.append(
        {
            "model": name,
            **calculate_metrics(y_true, predictions),
        }
    )

df_test_results = pd.DataFrame(test_results)

print("\n=== Résultats du test indépendant ===")
display(df_test_results)

# Gain de la Random Forest face à la référence hebdomadaire.
baseline_mae = df_test_results.loc[
    df_test_results["model"] == "Baseline H-168",
    "mae_kw",
].iloc[0]

rf_mae = df_test_results.loc[
    df_test_results["model"] == "Random Forest 365j",
    "mae_kw",
].iloc[0]

improvement_pct = (baseline_mae - rf_mae) / baseline_mae * 100

print(f"\nAmélioration MAE vs H-168 : {improvement_pct:.2f} %")

=== Couverture du test ===
Exemples de test avec H-168 : 361,778
Exemples communs H-24 / H-168 : 360,284

=== Résultats du test indépendant ===


,model,mae_kw,rmse_kw,wape_pct
0,Baseline H-24,41.309873,121.314683,10.709395
1,Baseline H-168,7.170016,11.379873,1.858794
2,Random Forest 365j,5.082631,7.986849,1.317649



Amélioration MAE vs H-168 : 29.11 %



## E — Conclusion générale et perspectives

### E.1 — Synthèse de l'expérimentation

Cette neuvième expérimentation clôture la phase exploratoire du projet Machine Learning de Smart Energy Optimizer.

Son objectif était d'étudier l'influence de la profondeur historique sur les performances d'un modèle de prévision de consommation énergétique, à partir des readings fournis par l'API EnerVision.

Contrairement aux premières expérimentations, qui reposaient sur des périodes relativement courtes ou sur un dataset historique distinct, cette étude exploite directement les données destinées à alimenter le futur pipeline du projet.

La collecte a permis de constituer un historique de 546 jours, couvrant la période du 1er janvier 2025 au 30 juin 2026, pour sept sites énergétiques.

Au total, **1 100 736 readings** ont été collectés, puis consolidés et analysés.

L'audit initial a confirmé une couverture temporelle complète, sans doublon ni intervalle irrégulier. Il a également identifié 65 275 consommations manquantes, soit 5,93 % des mesures.

L'exploitation des indicateurs `data_quality` et `null_reasons` a permis de relier ces absences à des défaillances déclarées, notamment des pannes de capteurs et des pertes de communication.

Les données ont ensuite été agrégées à une granularité horaire, en imposant un minimum de neuf mesures valides sur douze.

Cette opération a produit **91 728 observations horaires**, dont **91 331 exploitables**, soit 99,57 %.

La construction des variables historiques et des horizons H+1 à H+24 a finalement permis de constituer un dataset supervisé de **2 189 893 exemples**.

### E.2 — Influence de la profondeur historique

Quatre modèles Random Forest ont été entraînés sur des fenêtres historiques de 30, 90, 180 et 365 jours.

Afin de limiter les facteurs de variation, les quatre modèles ont conservé les mêmes caractéristiques, les mêmes hyperparamètres et une période de validation commune, de janvier à mars 2026.

Les résultats obtenus sont les suivants :

| Historique | MAE | RMSE | WAPE | Temps d'entraînement |
|---|---:|---:|---:|---:|
| 30 jours | 12,06 kW | 19,49 kW | 3,23 % | 1,66 s |
| 90 jours | 14,70 kW | 23,66 kW | 3,93 % | 5,46 s |
| 180 jours | 11,91 kW | 20,18 kW | 3,19 % | 13,80 s |
| **365 jours** | **5,49 kW** | **9,20 kW** | **1,47 %** | **32,31 s** |

La fenêtre de 365 jours obtient les meilleures performances, avec une réduction d'environ **54,5 % de la MAE** par rapport à la fenêtre de 30 jours.

L'amélioration n'est toutefois pas proportionnelle à la quantité de données : la fenêtre de 90 jours présente des performances inférieures à celle de 30 jours.

Ces résultats suggèrent que la diversité temporelle des observations joue un rôle important dans la capacité de généralisation du modèle.

La présence d'un cycle annuel complet pourrait notamment permettre de mieux représenter les variations saisonnières de consommation. Cette hypothèse ne peut cependant pas être isolée de l'effet de l'augmentation du volume d'entraînement.

Malgré un volume supérieur à 1,4 million d'exemples, le temps d'entraînement du modèle annuel reste limité à environ 32 secondes dans notre environnement expérimental.

**La fenêtre de 365 jours est donc retenue comme configuration de référence pour la suite du projet.**

### E.3 — Validation sur une période indépendante

Le modèle sélectionné a ensuite été évalué sur une période indépendante, du 1er avril au 30 juin 2026.

Cette période n'a été utilisée ni pour l'entraînement ni pour la sélection de la profondeur historique.

L'évaluation repose sur 360 284 exemples communs, disposant des références quotidiennes et hebdomadaires nécessaires à la comparaison.

Trois approches ont été évaluées :

| Approche | MAE | RMSE | WAPE |
|---|---:|---:|---:|
| Baseline H-24 | 41,31 kW | 121,31 kW | 10,71 % |
| Baseline H-168 | 7,17 kW | 11,38 kW | 1,86 % |
| **Random Forest 365 jours** | **5,08 kW** | **7,99 kW** | **1,32 %** |

La Random Forest obtient les meilleures performances sur les trois métriques.

Elle réduit notamment la MAE de **29,11 % par rapport à la baseline hebdomadaire H-168**.

Ce résultat confirme que le modèle ne se contente pas de reproduire la consommation de la semaine précédente : il exploite les autres caractéristiques disponibles pour améliorer les prévisions.

La MAE observée sur le test indépendant (5,08 kW) est également proche, et même légèrement inférieure, à celle obtenue sur la validation (5,49 kW).

Cette observation est encourageante, mais ne constitue pas à elle seule une garantie de stabilité à long terme.

### E.4 — Limites et points de vigilance

Malgré ces résultats, plusieurs limites doivent être prises en considération.

**Nature des données**

Les readings utilisés proviennent de l'environnement EnerVision étudié et concernent uniquement sept sites. Les performances obtenues ne garantissent pas une généralisation à de nouveaux sites ou à des données énergétiques réelles présentant des comportements différents.

**Gestion des données manquantes**

Le modèle repose notamment sur la disponibilité de la référence hebdomadaire `target_lag_168h`.

Les observations ne disposant pas de cette référence ont été exclues des expérimentations comparatives. Le futur service de prédiction devra donc prévoir une stratégie de repli lorsque l'historique nécessaire est indisponible.

**Qualité des données horaires**

Le seuil de neuf readings valides par heure permet de conserver une grande partie des observations, mais les moyennes calculées à partir de mesures partielles peuvent rester moins représentatives qu'une heure complète.

**Profondeur historique**

Les résultats démontrent l'intérêt de la fenêtre annuelle parmi les quatre configurations étudiées. Ils ne permettent pas d'affirmer que 365 jours constituent une durée universellement optimale.

**Évaluation temporelle**

Le jeu de test indépendant valide les performances sur une période future de trois mois et sur les mêmes sept sites. Des évaluations complémentaires seront nécessaires pour suivre la stabilité des performances dans le temps.

### E.5 — Enseignements et décisions techniques

Les expérimentations conduites au cours de cette phase exploratoire permettent de retenir plusieurs décisions pour le futur service ML :

1. **Granularité horaire :** les readings de cinq minutes seront agrégés en observations horaires, avec un contrôle explicite de leur qualité.

2. **Prévision multi-horizon :** le modèle produira des prévisions de consommation de H+1 à H+24.

3. **Architecture du modèle :** une Random Forest globale sera utilisée comme première référence opérationnelle, commune aux sept sites.

4. **Historique d'entraînement :** une fenêtre glissante de 365 jours sera privilégiée pour les premiers entraînements.

5. **Caractéristiques prédictives :** les variables calendaires, les consommations historiques et la référence hebdomadaire de l'heure cible seront conservées.

6. **Gestion des indisponibilités :** les consommations manquantes ne seront pas assimilées à des consommations nulles. Les règles de préparation devront préserver la continuité temporelle et prévoir une stratégie de repli en inférence.

7. **Évaluation :** les métriques MAE, RMSE et WAPE seront utilisées pour comparer les modèles, en complément des baselines historiques.

Ces décisions constituent une configuration initiale validée expérimentalement, susceptible d'évoluer lors des prochaines itérations.

### E.6 — Transition vers l'industrialisation et le MLOps

La phase exploratoire a permis de démontrer la faisabilité de la prévision énergétique multi-horizon et de sélectionner une première configuration de modèle.

La prochaine étape consistera à transformer les traitements expérimentaux en un pipeline Machine Learning reproductible, maintenable et intégrable à l'architecture de Smart Energy Optimizer.

Les travaux porteront notamment sur :

- L'extraction des traitements de préparation et de feature engineering vers des modules Python réutilisables.
- L'automatisation des entraînements et des évaluations.
- L'intégration de MLflow pour tracer les paramètres, métriques, versions de modèles et artefacts.
- L'expérimentation de nouvelles configurations et l'optimisation des hyperparamètres sur des jeux de validation dédiés.
- La mise en place d'une stratégie de sélection et de gestion des modèles.
- L'exposition des prévisions à travers une API REST FastAPI.
- L'intégration progressive avec les données préparées dans le Data Lake AWS.
- La mise en place de contrôles permettant de suivre la qualité des données et les performances du modèle dans le temps.

Le modèle Random Forest entraîné sur 365 jours constituera la première référence de ce pipeline.

### E.7 — Conclusion

Cette expérimentation apporte une réponse concrète à la question initiale : **sur les readings EnerVision étudiés, l'utilisation d'un historique annuel améliore significativement les performances de prévision par rapport aux fenêtres plus courtes évaluées.**

L'évaluation finale confirme également la valeur ajoutée du Machine Learning par rapport à une simple reconduction des consommations historiques.

Avec une MAE de 5,08 kW, un WAPE de 1,32 % et une amélioration de 29,11 % par rapport à la baseline hebdomadaire sur le jeu de test indépendant, la Random Forest globale constitue une base prometteuse pour le MVP.

Ces résultats ne signifient pas que le modèle est prêt à être déployé sans adaptations. Ils démontrent cependant que l'approche retenue mérite d'être industrialisée.

**La phase exploratoire est donc considérée comme achevée. La suite du projet sera consacrée à la transformation de ce prototype expérimental en un véritable service Machine Learning, intégrant les principes de reproductibilité, d'automatisation et de suivi propres au MLOps.**
